# 10. Bucketed DV01

## Objective

This notebook demonstrates the calculation of key-rate (bucketed) DV01 for an Interest Rate Swap.

Unlike parallel DV01, bucketed DV01 measures the sensitivity of the swap to individual points on the yield curve.

The workflow is:

Market Quotes
→ Bootstrap Yield Curve
→ Build Interest Rate Swap
→ Price Swap
→ Shift One Curve Tenor
→ Reprice Swap
→ Compute Bucketed DV01

In [1]:
from datetime import date

from src.common.enums import (
    DayCountConvention,
    FloatingIndex,
    PayReceive,
    PaymentFrequency,
)

from src.curves.bootstrapper import Bootstrapper

from src.market_data.market_instrument import MarketInstrument
from src.market_data.market_quote_collection import MarketQuoteCollection

from src.products.trade import Trade
from src.products.fixed_leg import FixedLeg
from src.products.floating_leg import FloatingLeg
from src.products.interest_rate_swap import InterestRateSwap

from src.risk.bucketed_dv01_calculator import BucketedDV01Calculator

In [2]:
quotes = MarketQuoteCollection()

quotes.add(MarketInstrument("Deposit", "1M", 0.05))
quotes.add(MarketInstrument("Deposit", "3M", 0.05))
quotes.add(MarketInstrument("Deposit", "6M", 0.05))
quotes.add(MarketInstrument("Deposit", "1Y", 0.05))
quotes.add(MarketInstrument("Deposit", "2Y", 0.05))
quotes.add(MarketInstrument("Deposit", "3Y", 0.05))

curve = Bootstrapper(
    valuation_date=date(2026, 1, 1),
    market_quotes=quotes,
).build()

In [3]:
trade = Trade(
    trade_id="IRS001",
    counterparty="ABC",
    currency="USD",
    effective_date=date(2026, 1, 1),
    maturity_date=date(2029, 1, 1),
)

fixed_leg = FixedLeg(
    notional=1_000_000,
    fixed_rate=0.05,
    payment_frequency=PaymentFrequency.ANNUAL,
    day_count=DayCountConvention.ACT_365,
    pay_receive=PayReceive.RECEIVE,
)

floating_leg = FloatingLeg(
    notional=1_000_000,
    payment_frequency=PaymentFrequency.ANNUAL,
    day_count=DayCountConvention.ACT_365,
    pay_receive=PayReceive.PAY,
    floating_index=FloatingIndex.SOFR,
)

swap = InterestRateSwap(
    trade=trade,
    fixed_leg=fixed_leg,
    floating_leg=floating_leg,
)

In [4]:
result = BucketedDV01Calculator().calculate(
    swap,
    curve,
)

print(f"Trade ID      : {result.trade_id}")
print(f"Currency      : {result.currency}")
print(f"Present Value : {result.present_value:.2f}")
print()

print("Bucketed DV01")

for bucket in result.bucketed_dv01:
    print(f"{bucket.tenor:>4} : {bucket.sensitivity:12.4f}")

Trade ID      : IRS001
Currency      : USD
Present Value : 135956.65

Bucketed DV01
  1M :       0.0000
  3M :       0.0000
  6M :       0.0000
  1Y :      -4.7559
  2Y :      -9.0475
  3Y :     -12.9440


## Interpretation

Each bucket represents the change in swap value when a single point on the yield curve is shifted by one basis point.

Bucketed DV01 is widely used for:

- Key-rate risk analysis
- Hedging
- Scenario analysis
- Regulatory market risk (FRTB)